In [9]:
!pip install -U langchain langchain-classic langchain-community \
    langchain-huggingface langchain-chroma langchain-text-splitters \
    sentence-transformers huggingface_hub

  Using cached huggingface_hub-2.3.0-py3-none-any.whl.metadata (16 kB)


In [2]:
import os
from getpass import getpass

from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_classic.chains import RetrievalQAWithSourcesChain

In [3]:
import os
from getpass import getpass


# 1. Set Hugging Face token
token = getpass("Enter your Hugging Face token: ")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = token
os.environ["HF_TOKEN"] = token # Explicitly set HF_TOKEN for underlying huggingface_hub client

Enter your Hugging Face token: ··········


In [9]:


# 2. Load document using textLoader
loader = TextLoader(r"/content/Generative AI learning.txt")
documents = loader.load()


In [48]:
# 3. Split document into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=30
)
chunks = splitter.split_documents(documents)

In [49]:
# 4. Create embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="all-mpnet-base-v2"
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [50]:
# 5. Store vectors in Chroma
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [61]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)


In [62]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x7970c45eead0>, search_kwargs={'k': 5})

In [53]:
from langchain_huggingface import HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-1.5B-Instruct", # Changed to a generally available instruct model
    task="text-generation",  # Set task explicitly to text-generation
    max_new_tokens=256,
    do_sample=False,
)

In [54]:
# 8. Create the RAG chain
qa_chain = RetrievalQAWithSourcesChain.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever
)

In [55]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()

print("Hugging Face model loaded successfully!")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Hugging Face model loaded successfully!


In [63]:
from langchain_huggingface import HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-1.5B-Instruct",
    task="text-generation",
    max_new_tokens=256,
    do_sample=False,
)

qa_chain = RetrievalQAWithSourcesChain.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever
)

In [64]:

def ask_pdf(question):

    # 1. Retrieve relevant chunks from Chroma
    docs = retriever.invoke(question)

    # 2. Handle cases where no chunks are retrieved
    if not docs:
        return {
            "answer": "I could not find relevant information in the document.",
            "sources": [],
            "input_tokens": 0
        }

    # 3. Combine retrieved chunks
    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    # 4. Create the prompt messages
    messages = [
        {
            "role": "system",
            "content": (
                "You are a PDF question-answering assistant. "
                "Answer using only the provided context. "
                "If the answer is missing, say you do not know. "
                "Do not invent information. "
                "Give a concise and accurate answer."
            )
        },
        {
            "role": "user",
            "content": (
                f"Context:\n{context}\n\n"
                f"Question: {question}"
            )
        }
    ]

    # 5. Format the Qwen chat prompt
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # 6. Tokenize the prompt
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    input_tokens = inputs["input_ids"].shape[1]

    # 7. Check the model's configured context limit
    context_limit = model.config.max_position_embeddings
    max_new_tokens = 256

    if input_tokens + max_new_tokens > context_limit:
        return {
            "answer": (
                f"Input is too long: {input_tokens} tokens. "
                "Reduce the retrieved context and try again."
            ),
            "sources": [],
            "input_tokens": input_tokens
        }

    # 8. Generate the answer
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # 9. Decode only newly generated tokens
    new_tokens = output[0][input_tokens:]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    # 10. Collect source information
    sources = [
        {
            "source": doc.metadata.get("source", "Unknown"),
            "page": doc.metadata.get(
                "page", doc.metadata.get("page_number", "Unknown")
            )
        }
        for doc in docs
    ]

    # 11. Return the answer and debugging information
    return {
        "answer": answer,
        "sources": sources,
        "input_tokens": input_tokens
    }


In [65]:

result = ask_pdf(
    "What is tokenization"
)

print("Answer:")
print(result["answer"])

print("\nInput tokens:")
print(result["input_tokens"])

print("\nSources:")
for source in result["sources"]:
    print(source)


Answer:
Tokenization is the process of breaking down text into individual units called tokens, which can include words, parts of speech, or other linguistic elements. These tokens are then mapped to unique identifiers (token IDs) for processing by a language model. The goal of tokenization is to enable efficient representation and manipulation of textual data within the framework of natural language processing tasks.

Input tokens:
531

Sources:
{'source': '/content/Generative AI learning.txt', 'page': 'Unknown'}
{'source': '/content/Generative AI learning.txt', 'page': 'Unknown'}
{'source': '/content/Generative AI learning.txt', 'page': 'Unknown'}
{'source': '/content/Generative AI learning.txt', 'page': 'Unknown'}
{'source': '/content/Generative AI learning.txt', 'page': 'Unknown'}


In [59]:
docs = retriever.invoke(
    "What is tokenization?"
)

for i, doc in enumerate(docs, start=1):
    print(f"\n--- Retrieved Chunk {i} ---")
    print(doc.page_content[:1000])
    print("Metadata:", doc.metadata)


--- Retrieved Chunk 1 ---
11. Quick interview questions and answers
Q1. What is a token?
A token is a unit of text, such as a word, part of a word, or punctuation, that a language model processes.
Q2. What is tokenization?
Tokenization is the process of splitting text into tokens and mapping them to token IDs that a model can process.
Q3. Why is token count important in LLMs?
It affects context-window usage, API billing, computational requirements, and how much information fits into a prompt.
Metadata: {'source': '/content/Generative AI learning.txt'}

--- Retrieved Chunk 2 ---
11. Quick interview questions and answers
Q1. What is a token?
A token is a unit of text, such as a word, part of a word, or punctuation, that a language model processes.
Q2. What is tokenization?
Tokenization is the process of splitting text into tokens and mapping them to token IDs that a model can process.
Q3. Why is token count important in LLMs?
It affects context-window usage, API billing, computational r

In [67]:

question = evaluation_dataset[0]["question"]

docs = retriever.invoke(question)
response = ask_pdf(question)

print("QUESTION:\n", question)

print("\nREFERENCE ANSWER:\n",
      evaluation_dataset[0]["ground_truth"])

print("\nGENERATED ANSWER:\n",
      response["answer"])

print("\nRETRIEVED CONTEXT:")
for i, doc in enumerate(docs, start=1):
    print(f"\nChunk {i}:")
    print(doc.page_content[:1000])


QUESTION:
 What is retrieval-augmented generation?

REFERENCE ANSWER:
 RAG retrieves relevant information from external documents and uses it to generate an answer.

GENERATED ANSWER:
 Retrieval-Augmented Generation (RAG) is a technique where a machine learning model generates text based on a query, but also incorporates pre-existing knowledge from external sources such as databases or other models. This approach combines the strengths of both generative and discriminative methods, allowing for more nuanced and contextually relevant outputs.

RETRIEVED CONTEXT:

Chunk 1:
It evaluates possible segmentations using learned probabilities and chooses a likely token sequence.
Example:
lowest
Could be split as:
["low", "est"]
or as a different sequence depending on its vocabulary and probabilities.
Advantages
* Can represent text using different possible segmentations.
* Supports vocabulary optimization during training.
Example use: SentencePiece Unigram models, including tokenizers used with

In [34]:
question = "Summarize the entire document."

docs = retriever.invoke(question)

for i, doc in enumerate(docs, start=1):
    print(f"\n--- Chunk {i} ---")
    print(doc.page_content[:1500])
    print("Metadata:", doc.metadata)


--- Chunk 1 ---
Previous conversation
2,000
Relevant PDF content
4,000
Total before answer
7,000
Remaining for answer
1,000
If the model needs more than the available space, some content may need to be removed, shortened, or processed in smaller sections.
Important: The context window is not necessarily the number of tokens the model can generate as an answer. Input and output share a context budget, subject to the model's specific limits.
Chat-with-PDF example
You upload a 200-page PDF and ask:
Metadata: {'source': '/content/Generative AI learning.txt'}

--- Chunk 2 ---
Type 6: Unigram tokenizer
Unigram tokenization starts with a large collection of possible subword units and learns which vocabulary best represents text.
It evaluates possible segmentations using learned probabilities and chooses a likely token sequence.
Example:
lowest
Could be split as:
["low", "est"]
or as a different sequence depending on its vocabulary and probabilities.
Advantages
* Can represent text using diff

In [ ]:
pip install -U gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 2.6 MB/s eta 0:00:00
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 2.7.0
    Uninstalling gradio_client-2.7.0:
      Successfully uninstalled gradio_client-2.7.0
  Attempting uninstall: gradio
    Found existing installation: gradio 6.27.0
    Uninstalling gradio-6.27.0:
      Successfully uninstalled gradio-6.27.0


In [ ]:
import gradio as gr

def chat_with_pdf(question, history):
    if not question or not question.strip():
        return "Please enter a question.", ""

    try:
        # Call your existing RAG function
        result = ask_pdf(question)

        answer = result["answer"]

        # Format retrieved source information
        sources = result.get("sources", [])

        if sources:
            source_text = "\n".join(
                f"- File: {s.get('source', 'Unknown')}, "
                f"Page: {s.get('page', 'Unknown')}"
                for s in sources
            )
        else:
            source_text = "No source metadata available."

        return answer, source_text

    except Exception as e:
        return f"An error occurred: {str(e)}", ""


with gr.Blocks(title="Chat with Your Document") as demo:
    gr.Markdown("# Chat with Your Document")
    gr.Markdown(
        "Ask questions about your document using "
        "Hugging Face, LangChain, and Chroma."
    )

    question = gr.Textbox(
        label="Your Question",
        placeholder="What is Generative AI?",
        lines=2
    )

    submit_btn = gr.Button("Ask Question", variant="primary")

    answer = gr.Textbox(
        label="Generated Answer",
        lines=8
    )

    sources = gr.Textbox(
        label="Retrieved Sources",
        lines=4
    )

    submit_btn.click(
        fn=chat_with_pdf,
        inputs=[question, gr.State([])],
        outputs=[answer, sources]
    )

    question.submit(
        fn=chat_with_pdf,
        inputs=[question, gr.State([])],
        outputs=[answer, sources]
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://255d50b127a04fcc67.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
